In [ ]:
# GlobalGeoTree 데이터셋 분석: 지구 곳곳의 나무들을 탐험하는 초급 AI 기지개 켜기 🌿
# 이 데이터셋은 전 세계의 나무 종(species)을 식별하기 위해 위성 이미지(Sentinel-2)와 기후/지리 환경 변수(고도, 토양 수분 등)를 종합적으로 사용합니다.
# 우리의 목표: 가장 기본적인 지리적 패턴을 발견하는 '초급 지구생물다양성 탐험가'가 되어 봅시다!

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset, DatasetDict, IterableDataset
import random
import time

# --- 설정 영역 ---
DATASET_ID = "yann111/GlobalGeoTree"
SAMPLE_COUNT = 100  # 분석할 샘플의 개수 (너무 많으면 메모리가 터질 수 있어요!)
# -----------------

print("🌿 튜터 모드: 안녕하세요! 저와 함께 데이터 과학의 신비로운 세계를 탐험해 볼 준비가 되었나요? 😊")
print("✨ 오늘 우리가 탐험할 것은 전 세계의 나무 데이터를 담은 'GlobalGeoTree'입니다!")

# 1. 데이터셋 로드 (스트리밍 모드 시도 및 오류 처리)
dataset = None
try:
    print("\n🌍 Step 1: 데이터셋을 불러옵니다... (Streaming 모드로 빠르게 시도해 볼게요!)")
    # streaming=True로 설정하여 메모리 효율성을 극대화합니다.
    dataset = load_dataset(DATASET_ID, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드(Streaming)로 데이터셋 로드에 성공했습니다. 메모리 걱정을 덜었네요!")
except Exception as e:
    print(f"⚠️ [경고] 스트리밍 로드에 실패했습니다. ({e})")
    print("➡️ 대신, 작은 부분만 다운로드하여 진행하겠습니다. 너무 걱정 마세요! 이 정도면 충분해요!")
    try:
        # 스트리밍 실패 시, 일반 모드로 작은 분량을 다운로드합니다.
        dataset = load_dataset(DATASET_ID, split='train')
    except Exception as e_fallback:
        print(f"🚨 치명적인 에러 발생: 데이터셋 로드에 실패했습니다. {e_fallback}")
        exit()

# 2. 분석을 위한 샘플 데이터 준비 (핵심 원칙: 전체를 다 쓰지 않고, 재미있는 일부만 뽑아 쓰기!)
print(f"\n🔬 Step 2: 데이터셋에서 무작위로 상위 {SAMPLE_COUNT}개의 샘플을 추출합니다...")

# 스트리밍 모드와 일반 모드에 따라 데이터 처리 패턴을 다르게 적용합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset) 패턴을 사용합니다.
    # 데이터셋을 반복 가능한 이터레이터(iterator)로 변환합니다.
    sample_iterator = dataset.take(SAMPLE_COUNT)
    # 파이썬의 next() 함수를 사용하여 메모리에 로드할 수 있는 샘플 리스트로 만듭니다.
    sample_data_list = [next(sample_iterator) for _ in range(SAMPLE_COUNT)]
else:
    # 일반 데이터셋 (Dataset) 패턴을 사용합니다.
    sample_data_list = list(dataset.take(SAMPLE_COUNT))
    
print(f"✨ 준비 완료! 이제 {len(sample_data_list)}개의 샘플로 지구탐험을 시작할 수 있어요!")


# 3. 초급 분석 실습: 지리적 환경 패턴 탐색 (Feature Engineering & Simple Visualization)
print("\n==================================================================")
print("🌳 Step 3: [Mission 1] 지리적 환경 변수를 이용한 분포 탐색")
print("==================================================================")

# 필요한 데이터를 추출하여 Pandas DataFrame을 만듭니다. 분석을 쉽게 하기 위함이에요!
lat_list = []
lon_list = []
elevation_list = []
species_list = []

for sample in sample_data_list:
    try:
        # 데이터셋의 깊은 구조에서 필요한 정보만 쏙쏙 뽑아옵니다.
        lat_list.append(sample['features']['auxiliary']['latitude'])
        lon_list.append(sample['features']['auxiliary']['longitude'])
        elevation_list.append(sample['features']['auxiliary']['elevation'])
        # 분류 단계가 가장 쉬운 '종(species)' 이름을 추출합니다.
        species_list.append(sample['features']['text']['level3_species'])
    except KeyError:
        # 만약 어떤 샘플에 필요한 키가 없다면, 그 샘플은 건너뛰어 줍니다. (튼튼한 코딩의 기본!)
        continue

# Pandas DataFrame으로 변환하여 분석하기 가장 좋은 형태로 만듭니다.
df = pd.DataFrame({
    'latitude': lat_list,
    'longitude': lon_list,
    'elevation': elevation_list,
    'species': species_list
})

# 3-1. 기초 통계 분석
mean_elevation = df['elevation'].mean()
max_elevation = df['elevation'].max()
min_elevation = df['elevation'].min()

print(f"\n🗺️ 지역 분석 보고서:")
print(f"  - 평균 고도: 약 {mean_elevation:.2f} 미터 (세계적으로 골고루 분포된 것 같네요!)")
print(f"  - 발견된 최대 고도: {max_elevation} 미터 (와, 얼마나 높이가 다른지 상상되시나요?)")
print(f"  - 발견된 최소 고도: {min_elevation} 미터")

# 3-2. 시각화: 고도와 위치의 관계 (산봉우리와 평야를 한번 봐요!)
plt.figure(figsize=(10, 8))
plt.scatter(df['longitude'], df['latitude'], c=df['elevation'], cmap='viridis', alpha=0.6)
plt.colorbar(label='Elevation (meters)')
plt.title('Global Tree Occurrence Density Map by Elevation Gradient (Sample)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


# 4. 심화 분석 실습: 종별 생존 여건 패턴 탐색 (Classification/Pattern Mining)
print("\n==================================================================")
print("🌲 Step 4: [Mission 2] 고도가 높은 곳에 주로 사는 나무 찾기 (Pattern Mining)")
print("==================================================================")

# 고도가 높은 곳 (예: 1000m 이상)에 집중적으로 분포하는 나무 종을 골라볼까요?
HIGH_ALTITUDE_THRESHOLD = 1000
high_altitude_df = df[df['elevation'] >= HIGH_ALTITUDE_THRESHOLD]

if not high_altitude_df.empty:
    # 가장 많이 발견된 상위 3가지 고지대 종을 찾습니다.
    top_species = high_altitude_df['species'].value_counts().nlargest(3)

    print(f"\n⛰️ 고도 {HIGH_ALTITUDE_THRESHOLD}m 이상의 고지대에서 발견된 종 (Top 3):")
    for i, (species, count) in enumerate(top_species.items()):
        print(f"  - {i+1}위: '{species}' 종 ({count}개 발견)")
    
    # 4-1. Visualization: 상위 종의 분포 확인
    plt.figure(figsize=(12, 6))
    
    # 종별로 데이터를 필터링하여 히스토그램(또는 박스 플롯)을 그리는 것이 일반적이지만,
    # 여기서는 초보자를 위해 고지대 데이터를 사용한 간단한 히스토그램을 만듭니다.
    
    plt.hist(high_altitude_df['elevation'], bins=50, color='coral', alpha=0.7)
    plt.title('Elevation Distribution of High Altitude Samples (Sample)')
    plt.xlabel('Elevation (meters)')
    plt.ylabel('Frequency (Number of Samples)')
    plt.axvline(HIGH_ALTITUDE_THRESHOLD, color='r', linestyle='--', label='Threshold')
    plt.legend()
    plt.show()
else:
    print("\n😔 아쉽게도, 이 샘플에서는 설정한 고도 임계값 이상의 데이터가 발견되지 않았습니다. 임계값을 낮춰보세요!")


# 5. 마무리와 튜터 코멘트
print("\n==================================================================")
print("🎉 축하합니다! 지구생물다양성 탐험을 성공적으로 완료했습니다!")
print("==================================================================")
print("💡 [튜터 코멘트]: 우리는 단지 '랜덤 샘플' 100개를 사용했을 뿐이지만, 이미 지리적 패턴과 종의 분포 간의 연관성을 찾아냈어요.")
print("🔥 진짜 AI는 여기에 더 많은 데이터를 넣어 '이런 환경(고도, 토양수분, 기온)에서는 이 종이 살아남을 확률이 90% 이상이다!'라고 예측할 수 있게 됩니다. 정말 놀랍죠? 😉")